# RSNA Knee — inference speed-up benchmark

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-011** | `hlly34/rsna-knee-bench` | inference speed-up benchmark: old cache path vs streaming (threaded/raw DICOM reads, CPU/GPU overlap, one pass for all checkpoints) vs selective decode; EXP-009 weights |

Runs `scripts/bench_inference.py` on training DICOMs with the 5 EXP-009 checkpoints. Each variant is timed on its own fresh studies (no file-cache advantage), then all variants predict the same 58 gold studies and are compared with the original path.

| Variant | What |
|---|---|
| A_cache_full | original: npz cache for all studies, then one DataLoader pass per checkpoint |
| B_stream_full | + threaded/raw DICOM reads, CPU/GPU overlap, one data pass for all checkpoints |
| C_stream_selective | + decode only the ~72 sampled slices; intensity window from them |
| D_stream_selective_pct4 | as C, plus every 4th slice for the intensity window |

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-011", "notebook": "hlly34/rsna-knee-bench", "what": "inference speed-up benchmark: old cache path vs streaming (threaded/raw DICOM reads, CPU/GPU overlap, one pass for all checkpoints) vs selective decode; EXP-009 weights"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=6):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


In [ ]:
WHEELS = walk_find(lambda d, f: d if os.path.basename(d) == "wheels" and any(x.endswith(".whl") for x in f) else None)
if WHEELS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--find-links", WHEELS,
                    "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
CKPTS = []
walk_find(lambda d, f: CKPTS.extend(os.path.join(d, x) for x in sorted(f) if x.endswith(".pt")) and None)
print(len(CKPTS), "checkpoints:", CKPTS)
t0 = time.time()
subprocess.run([sys.executable, f"{CODE}/scripts/bench_inference.py", "--data_dir", COMP, "--ckpts", *CKPTS,
                "--n_time", "150", "--out", "/kaggle/working/bench_inference.json"],
               env=dict(os.environ, PYTHONPATH=CODE), check=True)
EXP.update(json.load(open("/kaggle/working/bench_inference.json")), minutes=round((time.time() - t0) / 60, 1))
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1)